# 06 - Event Study: Cannabis Legalization

Dynamic treatment effects (leads and lags) around the legalization date. Pre-period coefficients are the parallel trends test.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

FARS_FILE = "fars_state_year.parquet"
CDC_FILE  = "cdc_state_year.parquet"

# Load FARS panel
if not (DATA_DIR / FARS_FILE).exists():
    raise FileNotFoundError(
        f"{FARS_FILE} not found. Run:\n"
        "  python scripts/download_fars.py\n"
        "  python src/build_fars_panel.py"
    )
fars = pd.read_parquet(DATA_DIR / FARS_FILE)
leg  = pd.read_csv("../data/codebooks/state_legalization_dates.csv")
print(f"FARS panel: {fars.shape}  |  States: {fars['state'].nunique()}  |  Years: {sorted(fars['year'].dropna().unique()[:3])}...{sorted(fars['year'].dropna().unique()[-3:])}")

In [ ]:
from linearmodels.panel import PanelOLS

In [ ]:
primary = "total_fatalities_per_100k" if "total_fatalities_per_100k" in fars.columns else "total_fatalities"

fars_reg = fars.merge(leg[['state','retail_sales_year']], on='state', how='left')
never_treated = leg[leg['retail_sales_year'].isna()]['state'].tolist()

# Relative time: years since legalization
# Never-treated: relative time = NaN
fars_reg['rel_t'] = np.where(
    fars_reg['retail_sales_year'].notna(),
    fars_reg['year'] - fars_reg['retail_sales_year'],
    np.nan
)

WINDOW = 5
ALL_PERIODS = [k for k in range(-WINDOW, WINDOW+1) if k != -1]

# Build dummies: treated state AND rel_t == k
for k in ALL_PERIODS:
    col = f"dum_{k}"
    fars_reg[col] = (
        (fars_reg['retail_sales_year'].notna()) &
        (fars_reg['rel_t'].fillna(999).clip(-WINDOW,WINDOW) == k)
    ).astype(float)

dum_cols = [f"dum_{k}" for k in ALL_PERIODS]

# Use never-treated + treated states (stacked)
in_window = leg[leg['retail_sales_year'].between(2010,2022)]['state'].tolist()
es_data = fars_reg[fars_reg['state'].isin(in_window + never_treated)].copy()
es_idx  = es_data.set_index(['state','year'])

es_res = PanelOLS(
    es_idx[primary],
    es_idx[dum_cols],
    entity_effects=True,
    time_effects=True,
).fit(cov_type='clustered', cluster_entity=True)

es_df = pd.DataFrame({
    'rel_t': ALL_PERIODS,
    'coef':  es_res.params[dum_cols].values,
    'ci_lo': es_res.conf_int().loc[dum_cols,'lower'].values,
    'ci_hi': es_res.conf_int().loc[dum_cols,'upper'].values,
}).sort_values('rel_t')

print("Pre-period (parallel trends check):")
print(es_df[es_df['rel_t']<0][['rel_t','coef']].round(4).to_string(index=False))

## Event study plot

In [ ]:
fig, ax = plt.subplots(figsize=(11,5))
ax.fill_between(es_df['rel_t'], es_df['ci_lo'], es_df['ci_hi'], alpha=0.2, color='steelblue')
ax.plot(es_df['rel_t'], es_df['coef'], 'o-', ms=5, lw=1.8, c='steelblue')
ax.plot(-1, 0, 'ro', ms=8, zorder=5, label='Reference period (t = -1)')
ax.axvline(0, color='red', ls='--', lw=1.5, label='Legalization (retail sales begin)')
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel("Years Relative to Retail Sales Start")
ax.set_ylabel(f"Coefficient on {primary.replace('_',' ')}")
ax.set_title(f"Event Study: Cannabis Legalization -> {primary.replace('_',' ').title()}\nTWFE; clustered SEs (state); window = ±{WINDOW} years")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "06_event_study_legalization.png", bbox_inches='tight')
plt.show()

## Pre-trend test

In [ ]:
from scipy import stats
pre  = es_df[es_df['rel_t'] < 0]['coef'].values
t, p = stats.ttest_1samp(pre, 0)
print(f"Pre-period mean coefficient: {pre.mean():.4f}")
print(f"t-test vs zero: t = {t:.2f}, p = {p:.3f}")
print()
print("Key question: do fatality trends look parallel between treated and")
print("never-treated states in the years before legalization?")